# 2D RSM from the Eiger2 R 500K: .brml metadata + .gfrm frames (xrayutilities)

Geometry is the same as the LynxEye notebook: source on the left, detector on the right, ω = angle of incidence.

$$Q_x = R[\cos(2\theta-\omega)-\cos\omega], \qquad Q_z = R[\sin\omega+\sin(2\theta-\omega)]$$

Each Eiger pixel has its own 2θ (along the columns) and its own sideways angle γ (along the rows). xrayutilities handles both.

**What is different from `gao_sto03.1_rsm2d_xrayutilities_ordered`, and why it gave 0 points**
* **φ:** the scan was run at φ = 270°. With φ applied, xrayutilities puts the in-plane part of Q into **Qy ≈ −1.61** (not Qx). The fixed `qy_window=(-0.5, 0.5)` then threw away every pixel.
  The default now is `use_phi=False`, so φ only rotates Q within the surface plane:
  * **Qx** is the in-plane component along the scattering plane (> 0 for grazing exit, < 0 for grazing incidence).
  * **Qy** is the sideways component, out of the scattering plane.
  * `use_phi=True` gives the true sample frame instead.
* **Windows follow the data:** there are no hard-coded Q windows. The ranges come from the data (`auto_range`), and you can still pass `h_range` and `qz_range`.
* **Angles and calibration from the .brml:**
  * ω and 2θ for every step.
  * χ (Bruker's "Psi-H") — no `90 − header_chi` guess.
  * φ, wavelength and time per step.
  * The beam centre and distance. The `.gfrm` header CENTER/DISTANC are used by default (the direct-beam position in the frame's own pixel coordinates). The .brml calibration is printed next to them and used if the header has none (`prefer="brml"` forces it).
* **Frame order:** frames are sorted naturally, so `Frame_10` comes after `Frame_9`.
  In a continuous scan the header angles are the middle of each exposure, half a step away from the .brml step values.
  The frame-to-step mapping is therefore chosen as the one whose frames have the same total counts as the .brml stores for each step.
* **Orientation check:** `check_geometry` tests the detector column direction from the data itself. A sharp substrate peak must map to the same Q in every frame.
* **Plots:** no mathtext, so `%matplotlib widget` works. Hover shows Q and intensity; a click marks a point.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

## 1. Metadata from the .brml (angles per step, chi/phi, detector calibration)

In [ ]:
import os, re, zipfile
import xml.etree.ElementTree as ET
import numpy as np


def read_eiger_brml(path, raw_index=0):
    """Scan metadata for a 2D (Eiger) measurement. The frames themselves are separate
    .gfrm files; the .brml holds, per step: omega, 2theta, max and sum pixel count, plus
    chi (Bruker 'Psi-H'), phi, wavelength, time per step and the detector calibration."""
    with zipfile.ZipFile(path) as z:
        root = ET.fromstring(z.read(f"Experiment0/RawData{raw_index}.xml"))
    route = root.find(".//DataRoute")

    datums = np.array([[float(v) for v in d.text.split(",")] for d in route.iter("Datum")])

    # which <Datum> column holds what (by axis id / logic name)
    columns = {}
    for view in route.findall(".//DataViews/RawDataView"):
        start = int(view.get("Start"))
        varying = view.find("Varying")
        if varying is not None:
            for k, fd in enumerate(varying.findall("FieldDefinitions")):
                axis = fd.get("AxisId")
                if axis and (axis not in columns or varying.get("LogicName") == "ScanAxes"):
                    columns[axis] = start + k
        rec = view.find("Recording")
        if rec is not None and view.get("xsi:type", view.get("{http://www.w3.org/2001/XMLSchema-instance}type")) != "ExternallyRecordedRawDataView":
            columns[rec.get("LogicName")] = start

    fixed = {d.get("LogicName"): float(d.find("Position").get("Value"))
             for d in root.findall(".//FixedInformation/Drives/InfoData")
             if d.find("Position") is not None}

    def axis_values(axis):
        if axis in columns:
            return datums[:, columns[axis]]
        return np.full(len(datums), fixed[axis])

    # detector calibration (Eiger filter info): beam centre inside the stored frame (ROI)
    filt = route.find(".//RawDataView/Recording/FilterInfoList/Eiger2FilterInfoData")
    size = route.find(".//RawDataView[@MeasuredDataIndex='0']/Recording/Size")
    det = dict(n_cols=int(size.find("X").text), n_rows=int(size.find("Y").text), pixel_size=0.075)
    if filt is not None:
        roi = filt.find("ROIPolygon")
        det.update(
            center_col=-float(roi.find("X").text),      # ROI origin is given relative to the beam centre
            center_row=-float(roi.find("Y").text),
            distance=float(filt.find("CalibratedDistance").get("Value")),
        )

    wl = root.find(".//WaveLength")
    scan = route.find("ScanInformation")
    ext = [s.text for s in route.findall("ExtLocations/string")]
    return dict(
        omega=axis_values("Theta"), twotheta=axis_values("TwoTheta"),
        max_pixel=datums[:, columns["maxPixelCount"]] if "maxPixelCount" in columns else None,
        sum_pixel=datums[:, columns["Sum Pixel Count"]] if "Sum Pixel Count" in columns else None,
        time_per_step=datums[0, 0],
        chi=fixed.get("Chi", 0.0), phi=fixed.get("Phi", 0.0),
        wavelength=float(wl.get("Value")) if wl is not None else 1.5406,
        detector=det, scan_name=scan.get("VisibleName") if scan is not None else "",
        frame_folder=ext[0].replace("\\", "/").split("/")[0] if ext else None,
        name=os.path.basename(path),
    )


## 2. Frames (.gfrm) and matching them to scan steps

In [ ]:
import fabio


def natural_key(name):
    """'Frame_10' after 'Frame_9' (plain sorted() puts Frame_10 before Frame_2)."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", name)]


def list_frames(frame_dir):
    return [os.path.join(frame_dir, f)
            for f in sorted(os.listdir(frame_dir), key=natural_key) if f.lower().endswith(".gfrm")]


def read_gfrm(path):
    """data + header values. ANGLES in a Bruker frame header = 2theta, omega, phi, chi
    (header chi uses Bruker's Eulerian-cradle convention, flat sample = 90; we take chi from
    the .brml instead, so it is only used for a consistency check)."""
    img = fabio.open(path)
    h = img.header
    def nums(key):
        return [float(v) for v in str(h.get(key, "")).split()] if key in h else None
    ang = nums("ANGLES")
    return dict(
        data=img.data.astype(np.float64),
        twotheta=ang[0] if ang else None, omega=ang[1] if ang else None,
        phi=ang[2] if ang else None, chi_header=ang[3] if ang else None,
    )


def _header_field(frame_path, key):
    """Numbers of one header field exactly as written in the .gfrm (fast: reads only the header).
    Returns None if the field is absent - fabio would silently fill in defaults for some."""
    with open(frame_path, "rb") as f:
        text = f.read(512 * 15).decode("latin1", errors="replace")
    m = re.search(rf"{key}\s*:\s*([-0-9.eE+ ]+)", text)
    return [float(v) for v in m.group(1).split()] if m and m.group(1).split() else None


def match_frames_to_steps(frames, meta, n_check=25):
    """Step index (into the .brml arrays) for every frame file.

    In a continuous scan the angles in a frame header are the MIDDLE of the exposure, i.e. half
    a step away from the .brml step values, so 'nearest step' is ambiguous. Several candidate
    mappings are therefore tried (file order, header angle rounded, shifted by +-half a step) and
    the one whose frames have the same total counts as the .brml 'Sum Pixel Count' is used.
    Prints a single summary line."""
    n, n_steps = len(frames), len(meta["omega"])
    # scan axis = whichever of omega / 2theta actually moves
    use_tt = np.ptp(meta["twotheta"]) > np.ptp(meta["omega"])
    axis_b = meta["twotheta"] if use_tt else meta["omega"]
    step = np.median(np.diff(axis_b)) if n_steps > 1 else 0.0
    ang = [_header_field(f, "ANGLES") for f in frames]
    have_angles = all(a is not None for a in ang) and step != 0

    cands = {}
    if have_angles:
        axis_h = np.array([a[0] if use_tt else a[1] for a in ang])
        frac = (axis_h - axis_b[0]) / step                 # fractional step index of each frame
        off = float(np.median(frac - np.round(frac)))      # ~0: step values, ~+-0.5: mid-exposure
        if abs(off) < 0.25:
            cands["header angle (nearest step)"] = np.rint(frac)
        cands["header angle + half step"] = np.rint(frac + 0.5)
        cands["header angle - half step"] = np.rint(frac - 0.5)
    if n == n_steps:
        cands["file order"] = np.arange(n)
    if not cands:
        raise ValueError(f"cannot match {n} frames to {n_steps} steps: no ANGLES in the headers "
                         "and the number of frames differs from the number of steps")
    cands = {k: np.clip(v.astype(int), 0, n_steps - 1) for k, v in cands.items()}

    check = np.unique(np.linspace(0, n - 1, min(n_check, n)).astype(int))
    scores = {k: None for k in cands}
    if meta["sum_pixel"] is not None:
        sums = np.array([fabio.open(frames[k]).data.sum() for k in check], float)
        for name, idx in cands.items():
            scores[name] = int(np.isclose(sums, meta["sum_pixel"][idx[check]], rtol=1e-3, atol=1.0).sum())
        best = max(cands, key=lambda k: scores[k])          # ties -> first candidate listed
    else:
        best = next(iter(cands))
    steps = cands[best]

    msg = f"frames ↔ steps: using '{best}'"
    if have_angles:
        msg += ("; header angles are half a step from the .brml values (mid-exposure)" if abs(off) > 0.25
                else "; header angles equal the .brml step values")
    if scores[best] is not None:
        msg += f"; total counts agree with the .brml for {scores[best]}/{len(check)} spot-checked frames"
        if scores[best] < len(check):
            msg += "  <-- CHECK: not all frames match"
    print(msg)
    if len(np.unique(steps)) != n:
        print(f"warning: {n - len(np.unique(steps))} frames share a step with another frame")
    return steps


## 3. Geometry and Q conversion (xrayutilities)

In [ ]:
import xrayutilities as xu


def detector_geometry(meta, frame_path=None, prefer="header", **override):
    """Beam centre (row, col), distance [mm], frame size and pixel size.
    Default: CENTER / DISTANC written in the .gfrm header (the direct-beam position in the
    frame's own pixel coordinates); if absent, or with prefer="brml", the calibration block of
    the .brml (its row/column centre is an interpretation of the ROI origin and can be ~0.5 px
    off). Keyword overrides win over both."""
    det = dict(meta["detector"])
    det["source"] = "brml"
    if frame_path is not None:
        det["n_rows"], det["n_cols"] = read_gfrm(frame_path)["data"].shape
        center, dist_cm = _header_field(frame_path, "CENTER"), _header_field(frame_path, "DISTANC")
        b = meta["detector"]
        if center and "center_col" in b:
            print(f"beam centre (col, row):  .brml ({b['center_col']:.2f}, {b['center_row']:.2f})   "
                  f"gfrm header ({center[0]:.2f}, {center[1]:.2f})")
        if dist_cm and "distance" in b:
            print(f"distance:                .brml {b['distance']:.2f} mm   gfrm header {10 * dist_cm[0]:.2f} mm")
        if center and "center_col" in b and (abs(center[0] - b["center_col"]) > 1 or abs(center[1] - b["center_row"]) > 1):
            print("  note: header and .brml centres differ by > 1 px")
        if prefer == "header" or "center_col" not in det:
            if center:
                det["center_col"], det["center_row"] = center[0], center[1]
                det["source"] = "gfrm header"
            if dist_cm:
                det["distance"] = 10.0 * dist_cm[0]
        g = read_gfrm(frame_path)
        if g["chi_header"] is not None:
            print(f"chi: .brml Psi-H = {meta['chi']}°, frame header chi = {g['chi_header']}° "
                  f"(90 - header = {90 - g['chi_header']:.4f}°); the .brml value is used")
    det.update(override)
    if override:
        det["source"] += " + manual override"
    print(f"using: centre (col, row) = ({det['center_col']:.3f}, {det['center_row']:.3f}), "
          f"distance = {det['distance']:.3f} mm  [{det['source']}]")
    return det


def make_qconv(det, wavelength):
    """D8 geometry (same as the LynxEye notebook): sample omega 'y-', chi 'x+', phi 'z-',
    detector 2theta 'y-', primary beam along +x  ->  source left, detector right.
    Frame columns run along 2theta ('z+'); frame rows run sideways (gamma, 'y-')."""
    qconv = xu.QConversion(['y-', 'x+', 'z-'], ['y-'], [1, 0, 0], wl=wavelength)
    qconv.init_area(det.get("row_dir", "y-"), det.get("col_dir", "z+"),
                    cch1=det["center_row"], cch2=det["center_col"],
                    Nch1=det["n_rows"], Nch2=det["n_cols"], distance=det["distance"],
                    pwidth1=det["pixel_size"], pwidth2=det["pixel_size"])
    return qconv


def convert_frames(frames, meta, det, steps=None, use_phi=False, every=1, threshold=0,
                   count_time=None, verbose=True):
    """Convert frames to Q, keeping only pixels with counts > threshold (Eiger frames are sparse).
    use_phi=False -> 'diffraction-plane' frame: Qx = in-plane along the scattering plane
    (Qx > 0 grazing exit, < 0 grazing incidence), Qy = sideways (out of the scattering plane).
    use_phi=True  -> sample frame (phi rotates the in-plane part between Qx and Qy).
    Returns dict of 1D float32 arrays Qx, Qy, Qz, I (cps), step."""
    qconv = make_qconv(det, meta["wavelength"])
    if steps is None:
        steps = match_frames_to_steps(frames, meta)
    t = count_time or meta["time_per_step"]
    phi = meta["phi"] if use_phi else 0.0
    out = {k: [] for k in ("Qx", "Qy", "Qz", "I", "step")}
    idx = range(0, len(frames), every)
    for n, k in enumerate(idx):
        s = steps[k]
        data = read_gfrm(frames[k])["data"]
        mask = data > threshold
        if not mask.any():
            continue
        qx, qy, qz = qconv.area(meta["omega"][s], meta["chi"], phi, meta["twotheta"][s])
        out["Qx"].append(qx[mask].astype(np.float32)); out["Qy"].append(qy[mask].astype(np.float32))
        out["Qz"].append(qz[mask].astype(np.float32)); out["I"].append((data[mask] / t).astype(np.float32))
        out["step"].append(np.full(mask.sum(), s, np.int32))
        if verbose and (n % 500 == 0):
            print(f"  {n}/{len(idx)} frames")
    if not out["I"]:
        raise ValueError("no pixel above threshold in any frame")
    return {k: np.concatenate(v) for k, v in out.items()}


def check_geometry(frames, meta, det, steps, min_fraction=0.2, reference_Q=None):
    """Self-consistency test of the detector column direction, using the data only.
    In a scan, a sharp substrate peak sits at one fixed Q while omega and 2theta move, so in a
    correct geometry its brightest pixel maps to the SAME Q in every frame where it is visible.
    With the wrong column (2theta) direction it wanders by ~4x the omega step per frame.
    Uses all frames whose maximum is >= min_fraction of the brightest frame.
    (The row direction only flips the sign of Qy and cannot be tested this way.)"""
    maxima = meta["max_pixel"][steps] if meta["max_pixel"] is not None \
        else np.array([read_gfrm(f)["data"].max() for f in frames])
    use = np.where(maxima >= min_fraction * maxima.max())[0]
    if len(use) < 3:
        print(f"only {len(use)} frame(s) contain the peak - lower min_fraction or use a finer scan")
        return None
    peaks = []
    for k in use:
        data = read_gfrm(frames[k])["data"]
        r, c = np.unravel_index(np.argmax(data), data.shape)
        r0, c0 = max(r - 2, 0), max(c - 2, 0)
        w = data[r0:r + 3, c0:c + 3]
        rr, cc = np.mgrid[r0:r0 + w.shape[0], c0:c0 + w.shape[1]]
        peaks.append((steps[k], (rr * w).sum() / w.sum(), (cc * w).sum() / w.sum()))
    results = []
    for col_dir in ("z+", "z-"):
        qc = make_qconv(dict(det, col_dir=col_dir), meta["wavelength"])
        q = []
        for s, r, c in peaks:
            qx, qy, qz = qc.area(meta["omega"][s], meta["chi"], 0.0, meta["twotheta"][s])
            q.append((qx[int(round(r)), int(round(c))], qz[int(round(r)), int(round(c))]))
        q = np.array(q)
        spread = float(np.hypot(*q.std(axis=0)))
        results.append((spread, col_dir, q.mean(axis=0)))
    results.sort(key=lambda t: t[0])
    print(f"substrate peak followed through {len(peaks)} frames (ω {meta['omega'][peaks[0][0]]:.4f}..{meta['omega'][peaks[-1][0]]:.4f}°):")
    for spread, cd, mean in results:
        ref = f",  {np.hypot(mean[0] - reference_Q[0], mean[1] - reference_Q[1]):.4f} from reference" if reference_Q else ""
        print(f"  cols {cd}:  spread {spread:.5f} Å⁻¹,  mean (Qx, Qz) = ({mean[0]:+.4f}, {mean[1]:.4f}){ref}")
    print(f"-> col_dir='{results[0][1]}'" + ("" if results[1][0] > 2 * results[0][0] else "   (not decisive: spreads are similar)"))
    return results[0][1]


## 4. Expected reflection positions (same as the LynxEye notebook)

In [ ]:
def expected_reflection(hkl, a, c=None, b=None, wavelength=1.5406):
    h, k, l = hkl
    b = a if b is None else b
    c = a if c is None else c
    q_ip = 2 * np.pi * np.hypot(h / a, k / b)
    q_z = 2 * np.pi * l / c
    q = np.hypot(q_ip, q_z)
    theta = np.degrees(np.arcsin(q * wavelength / (4 * np.pi)))
    psi = np.degrees(np.arctan2(q_ip, q_z))
    return {"2theta": 2 * theta, "psi": psi, "Qz": q_z,
            "grazing_incidence": dict(omega=theta - psi, Qx=-q_ip, accessible=theta - psi > 0),
            "grazing_exit": dict(omega=theta + psi, Qx=q_ip, accessible=psi < theta),
            "skew_symmetric": dict(omega=theta, chi=psi, Qx=q_ip, accessible=True)}


def geometry_of_scan(meta, chi_threshold=2.0):
    if abs(meta["chi"]) > chi_threshold:
        return "skew_symmetric"
    return "grazing_exit" if np.median(meta["omega"] - meta["twotheta"] / 2) > 0 else "grazing_incidence"


## 5. Plotting (no mathtext -> safe with %matplotlib widget)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogLocator, FuncFormatter, NullFormatter
from scipy.stats import binned_statistic_2d
from scipy.ndimage import gaussian_filter

_SUP = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")


def _log_label(v, _pos=None):
    if v <= 0:
        return ""
    e = np.log10(v)
    return f"10{str(int(round(e))).translate(_SUP)}" if abs(e - round(e)) < 1e-6 else ""


def _is_widget_backend():
    b = matplotlib.get_backend().lower()
    return "ipympl" in b or "widget" in b


def auto_range(values, weights, frac=0.999, pad=0.15):
    """Range containing `frac` of the total intensity, padded - so the window follows the data
    instead of a hard-coded reflection-specific window."""
    order = np.argsort(values)
    cw = np.cumsum(weights[order]); cw /= cw[-1]
    lo = values[order][np.searchsorted(cw, (1 - frac) / 2)]
    hi = values[order][min(np.searchsorted(cw, 1 - (1 - frac) / 2), len(cw) - 1)]
    span = (hi - lo) or 1e-3
    return float(lo - pad * span), float(hi + pad * span)


def plot_rsm2d(Q, horizontal="Qx", h_range=None, qz_range=None, n_bins=400, statistic="max",
               smooth_sigma=0.0, vmin=None, title="", markers=None, equal_aspect=True,
               save=None, dpi=None):
    """Bin the converted pixels and plot `horizontal` (Qx, Qy or Qpar) against Qz.
    Ranges default to auto_range of the data. statistic: 'max', 'mean' or 'sum'."""
    H = np.hypot(Q["Qx"], Q["Qy"]) if horizontal == "Qpar" else Q[horizontal]
    h_range = h_range or auto_range(H, Q["I"])
    qz_range = qz_range or auto_range(Q["Qz"], Q["I"])
    sel = (H >= h_range[0]) & (H <= h_range[1]) & (Q["Qz"] >= qz_range[0]) & (Q["Qz"] <= qz_range[1])
    if not sel.any():
        raise ValueError(f"no data in {horizontal} {h_range}, Qz {qz_range}; "
                         f"data spans {horizontal} {H.min():.3f}..{H.max():.3f}, Qz {Q['Qz'].min():.3f}..{Q['Qz'].max():.3f}")
    grid, he, ze, _ = binned_statistic_2d(H[sel], Q["Qz"][sel], Q["I"][sel], statistic=statistic,
                                          bins=n_bins, range=[h_range, qz_range])
    if smooth_sigma > 0:   # smooth only where there is data (normalised convolution)
        w = np.isfinite(grid).astype(float)
        num = gaussian_filter(np.nan_to_num(grid) * w, smooth_sigma)
        den = gaussian_filter(w, smooth_sigma)
        with np.errstate(invalid="ignore", divide="ignore"):
            grid = np.where(den > 1e-6, num / den, np.nan)
    grid = np.where(grid > 0, grid, np.nan)
    hc, zc = 0.5 * (he[1:] + he[:-1]), 0.5 * (ze[1:] + ze[:-1])

    if dpi is None:
        dpi = 100 if _is_widget_backend() else 200
    fig, ax = plt.subplots(figsize=(5.5, 6), dpi=dpi, layout="constrained")
    ax.set_facecolor("black")
    vmax = np.nanmax(grid)
    vmin = vmin or max(np.nanmin(grid), vmax * 1e-6)
    im = ax.pcolormesh(hc, zc, grid.T, cmap="inferno", norm=LogNorm(vmin=vmin, vmax=vmax),
                       shading="auto", rasterized=True)
    for qh, qz, label in (markers or []):
        ax.plot(qh, qz, "+", color="cyan", ms=12, mew=1)
        ax.annotate(label, (qh, qz), xytext=(6, 6), textcoords="offset points", color="cyan", fontsize=7)
    if equal_aspect:
        ax.set_aspect("equal")
    label = {"Qpar": "Q∥"}.get(horizontal, horizontal)
    ax.set_xlabel(f"{label} / Å⁻¹"); ax.set_ylabel("Qz / Å⁻¹"); ax.set_title(title, fontsize=7)
    cb = fig.colorbar(im, ax=ax, label="Intensity (cps)", shrink=0.8)
    cb.ax.yaxis.set_major_locator(LogLocator(base=10))
    cb.ax.yaxis.set_major_formatter(FuncFormatter(_log_label))
    cb.ax.yaxis.set_minor_formatter(NullFormatter())

    def format_coord(x, y):
        i = np.clip(np.searchsorted(he, x) - 1, 0, n_bins - 1); j = np.clip(np.searchsorted(ze, y) - 1, 0, n_bins - 1)
        v = grid[i, j]
        return f"{label} = {x:.5f}, Qz = {y:.5f}   |  I = {'—' if not np.isfinite(v) else f'{v:.3g}'}"
    ax.format_coord = format_coord

    picked = []
    def on_click(event):
        if event.inaxes is not ax or (fig.canvas.toolbar is not None and fig.canvas.toolbar.mode):
            return
        for a in picked: a.remove()
        picked[:] = [ax.plot(event.xdata, event.ydata, "x", color="lime", ms=8)[0],
                     ax.annotate(f"{label} = {event.xdata:.4f}\nQz = {event.ydata:.4f}", (event.xdata, event.ydata),
                                 xytext=(8, -22), textcoords="offset points", color="lime", fontsize=7)]
        fig.canvas.draw_idle()
    fig.canvas.mpl_connect("button_press_event", on_click)
    if save:
        fig.savefig(save, dpi=300, bbox_inches="tight", facecolor="black")
    plt.show()
    return fig, ax, (hc, zc, grid)


def plot_frame(frame_path, meta, det, step, use_phi=False):
    """Single frame: raw detector image (left) and its pixels in Qx-Qz (right)."""
    g = read_gfrm(frame_path)
    data = g["data"]
    qconv = make_qconv(det, meta["wavelength"])
    qx, qy, qz = qconv.area(meta["omega"][step], meta["chi"], meta["phi"] if use_phi else 0.0, meta["twotheta"][step])
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4), dpi=100 if _is_widget_backend() else 150, layout="constrained")
    pos = data[data > 0]
    norm = LogNorm(vmin=1, vmax=max(pos.max(), 2)) if pos.size else None
    im = a1.imshow(np.where(data > 0, data, np.nan), origin="lower", cmap="inferno", norm=norm, aspect="auto")
    a1.plot(det["center_col"], det["center_row"], "c+", ms=12)
    a1.set_xlabel("column (2θ direction)"); a1.set_ylabel("row"); a1.set_facecolor("black")
    a1.set_title(f"{os.path.basename(frame_path)}  ω = {meta['omega'][step]:.4f}°, 2θ = {meta['twotheta'][step]:.4f}°", fontsize=8)
    m = data > 0
    a2.set_facecolor("black")
    if m.any():
        o = np.argsort(data[m])
        a2.scatter(qx[m][o], qz[m][o], c=data[m][o], s=2, cmap="inferno", norm=norm)
    a2.set_xlabel("Qx / Å⁻¹"); a2.set_ylabel("Qz / Å⁻¹"); a2.set_title("same pixels in Q", fontsize=8)
    for cb_ax in (a1,):
        cb = fig.colorbar(im, ax=cb_ax, label="counts")
        cb.ax.yaxis.set_major_formatter(FuncFormatter(_log_label)); cb.ax.yaxis.set_minor_formatter(NullFormatter())
    plt.show()
    return fig


## 6. Run: STO (103), grazing exit

In [ ]:
brml_path = r"C:\Users\j.bantol\Documents\Data\RSM\IHE-1\2026-11-12_Cu_gao_sto03.1\RSM_103p_2Dsnapshot_om-tth_56.8-57.2deg_0.0001deg_0.5s.brml"
frame_dir = r"C:\Users\j.bantol\Documents\Data\RSM\IHE-1\2026-11-12_Cu_gao_sto03.1\RSM_103p_2Dsnapshot_om-tth_56.8-57.2deg_0.0001deg_0.5s_FrameFiles\frames"
STO = dict(a=3.905, label="STO")
hkl = (1, 0, 3)

meta = read_eiger_brml(brml_path)
print(f"{meta['scan_name']}: {len(meta['omega'])} steps, ω {meta['omega'][0]}..{meta['omega'][-1]}°, "
      f"2θ {meta['twotheta'][0]}..{meta['twotheta'][-1]}°, χ = {meta['chi']}°, φ = {meta['phi']}°, "
      f"{meta['time_per_step']} s/step, {geometry_of_scan(meta)}")

In [ ]:
frames = list_frames(frame_dir)
print(f"{len(frames)} frames: {os.path.basename(frames[0])} ... {os.path.basename(frames[-1])}")
steps = match_frames_to_steps(frames, meta)          # one summary line incl. the total-count check
det = detector_geometry(meta, frames[0])              # .gfrm header centre/distance (prints the .brml values next to them)
# manual override if needed, e.g.: det = detector_geometry(meta, frames[0], center_row=226.5, center_col=494.03)

In [ ]:
# which way do the columns run? (decided from the substrate peak; set it once it is clear)
ref = expected_reflection(hkl, STO["a"], wavelength=meta["wavelength"])
geom = geometry_of_scan(meta)
det["col_dir"] = check_geometry(frames, meta, det, steps, reference_Q=(ref[geom]["Qx"], ref["Qz"])) or "z+"

In [ ]:
%matplotlib inline
plt.close("all")
k_peak = int(np.argmax(meta["max_pixel"][steps])) if meta["max_pixel"] is not None else len(frames) // 2
plot_frame(frames[k_peak], meta, det, steps[k_peak]);

In [ ]:
# full conversion (4001 frames: a few minutes; every=4 for a quick look)
Q = convert_frames(frames, meta, det, steps=steps, use_phi=False, every=1)
print(f"{len(Q['I'])} pixels with counts;  Qx {Q['Qx'].min():.3f}..{Q['Qx'].max():.3f},  "
      f"Qy {Q['Qy'].min():.3f}..{Q['Qy'].max():.3f},  Qz {Q['Qz'].min():.3f}..{Q['Qz'].max():.3f} Å⁻¹")
k = np.argmax(Q["I"])
print(f"brightest pixel: Qx = {Q['Qx'][k]:+.4f}, Qy = {Q['Qy'][k]:+.4f}, Qz = {Q['Qz'][k]:.4f} Å⁻¹   "
      f"(bulk STO {''.join(map(str, hkl))}: Qx = {ref[geom]['Qx']:+.4f}, Qz = {ref['Qz']:.4f})")

In [ ]:
%matplotlib inline
plt.close("all")
mark = [(ref[geom]["Qx"], ref["Qz"], f"STO {''.join(map(str, hkl))}")]
plot_rsm2d(Q, "Qx", title="STO (103) grazing exit - scattering plane", markers=mark,
           statistic="max", smooth_sigma=1.0, save="rsm_STO103_Qx.png")
plot_rsm2d(Q, "Qy", title="STO (103) grazing exit - out of plane", markers=[(0.0, ref["Qz"], "")],
           statistic="max", smooth_sigma=1.0, save="rsm_STO103_Qy.png");

In [ ]:
%matplotlib widget
plt.close("all")
# zoom on the peak, e.g. h_range=(1.58, 1.64), qz_range=(4.78, 4.86)
plot_rsm2d(Q, "Qx", markers=mark, statistic="max", equal_aspect=False);